# Preparação do Dataset Analítico

Este notebook tem como objetivo consolidar as bases da camada Gold em um único dataset analítico para as etapas de Análise Exploratória de Dados (EDA) e Machine Learning.

Serão utilizados dois conjuntos principais:

- `gold_alunos.parquet`: dados em nível individual;
- `gold_municipal.parquet`: dados agregados em nível municipal.

Ao longo do notebook serão realizadas:

- análise estrutural das bases;
- validação de granularidade;
- análise de qualidade;
- identificação das chaves de integração;
- avaliação da relevância das variáveis;
- identificação de possíveis riscos de data leakage;
- seleção das variáveis utilizadas no dataset consolidado;
- integração das bases;
- validação do dataset final.

O objetivo é produzir uma base única, consistente e documentada para as próximas etapas do projeto.

## 1. Importação das bibliotecas

Inicialmente são carregadas as bibliotecas utilizadas para manipulação, validação e análise estrutural dos dados.

In [4]:
from pathlib import Path

import numpy as np
import pandas as pd

## 2. Configurações gerais

São definidas configurações de visualização do Pandas para facilitar a inspeção dos datasets durante o processo de preparação.

In [2]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 3. Definição dos caminhos

Os datasets utilizados neste notebook estão armazenados na camada Gold do projeto.

In [5]:
BASE_DIR = Path("..")

DATA_DIR = BASE_DIR / "data"
GOLD_DIR = DATA_DIR / "gold"

ALUNOS_PATH = GOLD_DIR / "gold_alunos.parquet"
MUNICIPAL_PATH = GOLD_DIR / "gold_municipal.parquet"

## 4. Carregamento dos datasets

São carregadas as duas principais bases analíticas disponíveis.

A base de alunos possui granularidade individual, enquanto a base municipal contém informações agregadas relacionadas aos indicadores de alfabetização e metas educacionais.

In [6]:
df_alunos = pd.read_parquet(ALUNOS_PATH)
df_municipal = pd.read_parquet(MUNICIPAL_PATH)

print(f"Alunos: {df_alunos.shape[0]:,} linhas | {df_alunos.shape[1]} colunas")
print(f"Municipal: {df_municipal.shape[0]:,} linhas | {df_municipal.shape[1]} colunas")

Alunos: 57,782 linhas | 16 colunas
Municipal: 23,995 linhas | 51 colunas


## 5. Visão inicial dos dados

Antes de realizar qualquer seleção ou integração, é importante conhecer a estrutura das duas bases e compreender quais informações estão disponíveis em cada uma.

In [7]:
display(df_alunos.head())

,ano,id_municipio,id_municipio_nome,id_escola,id_aluno,caderno,serie,rede,presenca,preenchimento_caderno,alfabetizado,proficiencia,peso_aluno,sigla_uf,meta_municipio_ano,meta_uf_ano
0,2023,1302504,Manacapuru,60000951,13015851,1,2° ano do Ensino Fundamental,Municipal,Ausente,Prova não preenchida,Não,NaN,NaN,<NA>,NaN,NaN
1,2023,1302603,Manaus,60000963,13030738,1,2° ano do Ensino Fundamental,Municipal,Ausente,Prova não preenchida,Não,NaN,NaN,<NA>,NaN,NaN
2,2023,1300631,Beruri,60001351,13003982,1,2° ano do Ensino Fundamental,Municipal,Ausente,Prova não preenchida,Não,NaN,NaN,<NA>,NaN,NaN
3,2023,1711506,Jaú do Tocantins,60004115,17012510,1,2° ano do Ensino Fundamental,Municipal,Ausente,Prova não preenchida,Não,NaN,NaN,<NA>,NaN,NaN
4,2023,2100709,Anajatuba,60004434,21012344,1,2° ano do Ensino Fundamental,Municipal,Ausente,Prova não preenchida,Não,NaN,NaN,<NA>,NaN,NaN


In [8]:
display(df_municipal.head())

,ano,id_municipio,serie,rede,taxa_alfabetizacao,media_portugues,proporcao_aluno_nivel_0,proporcao_aluno_nivel_1,proporcao_aluno_nivel_2,proporcao_aluno_nivel_3,proporcao_aluno_nivel_4,proporcao_aluno_nivel_5,proporcao_aluno_nivel_6,proporcao_aluno_nivel_7,proporcao_aluno_nivel_8,ano_meta_municipio,taxa_alfabetizacao_meta_municipio,meta_alfabetizacao_2024,meta_alfabetizacao_2025,meta_alfabetizacao_2026,meta_alfabetizacao_2027,meta_alfabetizacao_2028,meta_alfabetizacao_2029,meta_alfabetizacao_2030,nivel_alfabetizacao,percentual_participacao,ano_meta_uf,sigla_uf,taxa_alfabetizacao_meta_uf,meta_alfabetizacao_2024_meta_uf,meta_alfabetizacao_2025_meta_uf,meta_alfabetizacao_2026_meta_uf,meta_alfabetizacao_2027_meta_uf,meta_alfabetizacao_2028_meta_uf,meta_alfabetizacao_2029_meta_uf,meta_alfabetizacao_2030_meta_uf,percentual_participacao_meta_uf,ano_meta_brasil,taxa_alfabetizacao_meta_brasil,meta_alfabetizacao_2024_meta_brasil,meta_alfabetizacao_2025_meta_brasil,meta_alfabetizacao_2026_meta_brasil,meta_alfabetizacao_2027_meta_brasil,meta_alfabetizacao_2028_meta_brasil,meta_alfabetizacao_2029_meta_brasil,meta_alfabetizacao_2030_meta_brasil,percentual_participacao_meta_brasil,meta_municipio_ano,meta_uf_ano,gap_meta,atingiu_meta
0,2023,1100031,2,Municipal,69.10,767.88,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"2,024.00",75.88,70.85,72.53,74.15,75.71,77.21,78.64,80.00,4.00,92.59,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,2023,1100072,2,Municipal,58.20,747.89,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"2,023.00",58.20,61.82,65.31,68.64,71.79,74.74,77.48,80.00,2.00,92.25,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
2,2023,1100189,2,Privada,69.73,762.41,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
3,2023,1101609,2,Municipal,50.70,745.68,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"2,024.00",78.73,55.53,60.25,64.80,69.09,73.07,76.71,80.00,4.00,91.20,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
4,2023,1101807,2,Municipal,55.69,752.37,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"2,024.00",44.06,59.72,63.63,67.37,70.89,74.18,77.22,80.00,1.00,88.64,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0


In [9]:
print("Colunas - Alunos")
print(df_alunos.columns.tolist())

print("\nColunas - Municipal")
print(df_municipal.columns.tolist())

Colunas - Alunos
['ano', 'id_municipio', 'id_municipio_nome', 'id_escola', 'id_aluno', 'caderno', 'serie', 'rede', 'presenca', 'preenchimento_caderno', 'alfabetizado', 'proficiencia', 'peso_aluno', 'sigla_uf', 'meta_municipio_ano', 'meta_uf_ano']

Colunas - Municipal
['ano', 'id_municipio', 'serie', 'rede', 'taxa_alfabetizacao', 'media_portugues', 'proporcao_aluno_nivel_0', 'proporcao_aluno_nivel_1', 'proporcao_aluno_nivel_2', 'proporcao_aluno_nivel_3', 'proporcao_aluno_nivel_4', 'proporcao_aluno_nivel_5', 'proporcao_aluno_nivel_6', 'proporcao_aluno_nivel_7', 'proporcao_aluno_nivel_8', 'ano_meta_municipio', 'taxa_alfabetizacao_meta_municipio', 'meta_alfabetizacao_2024', 'meta_alfabetizacao_2025', 'meta_alfabetizacao_2026', 'meta_alfabetizacao_2027', 'meta_alfabetizacao_2028', 'meta_alfabetizacao_2029', 'meta_alfabetizacao_2030', 'nivel_alfabetizacao', 'percentual_participacao', 'ano_meta_uf', 'sigla_uf', 'taxa_alfabetizacao_meta_uf', 'meta_alfabetizacao_2024_meta_uf', 'meta_alfabetizac

## 6. Análise estrutural

Nesta etapa será realizada uma análise estrutural dos datasets, considerando:

- quantidade de registros;
- quantidade de colunas;
- tipos dos dados;
- valores ausentes;
- cardinalidade;
- duplicidades.

O objetivo não é realizar ainda a análise exploratória completa, mas verificar se as bases estão adequadas para integração e construção do dataset analítico.

In [10]:
def resumo_estrutura(df):
    return pd.DataFrame({
        "tipo": df.dtypes.astype(str),
        "nulos": df.isna().sum(),
        "pct_nulos": (df.isna().mean() * 100).round(2),
        "valores_unicos": df.nunique(dropna=True)
    })

In [11]:
display(
    resumo_estrutura(df_alunos)
)

,tipo,nulos,pct_nulos,valores_unicos
ano,int64,0,0.00,2
id_municipio,string,0,0.00,4591
id_municipio_nome,string,0,0.00,4397
id_escola,string,0,0.00,24346
id_aluno,string,0,0.00,57449
caderno,int64,0,0.00,4
serie,string,0,0.00,1
rede,string,0,0.00,2
presenca,string,0,0.00,2
preenchimento_caderno,string,0,0.00,2


In [12]:
display(
    resumo_estrutura(df_municipal)
)

,tipo,nulos,pct_nulos,valores_unicos
ano,int64,0,0.00,2
id_municipio,string,0,0.00,5550
serie,int64,0,0.00,1
rede,string,0,0.00,4
taxa_alfabetizacao,float64,0,0.00,6161
media_portugues,float64,0,0.00,13321
proporcao_aluno_nivel_0,float64,11547,48.12,943
proporcao_aluno_nivel_1,float64,11547,48.12,1374
proporcao_aluno_nivel_2,float64,11547,48.12,1922
proporcao_aluno_nivel_3,float64,11547,48.12,2289
